In [1]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime as dt
import xarray as xr
import os
import matplotlib as mpl
fs=10
mpl.rc('xtick', labelsize=fs)
mpl.rc('ytick', labelsize=fs)
mpl.rc('legend', fontsize=fs)
mpl.rc('axes', titlesize=fs)
mpl.rc('axes', labelsize=fs)
mpl.rc('figure', titlesize=fs)
mpl.rc('font', size=fs)
mpl.rc('font', family='sans-serif', weight='normal', style='normal')



In [2]:
def make_filename(path_run,folder, var='biol_T', res='d'):
    """Construct path prefix for local SHEM results given date object and paths dict"""
    prefix = os.path.join(path_run, f'{folder}/')
    fname = []
    for file in os.listdir(prefix):
        if (var in file) and ('_1'+res) in file:
            fname.append(file)
    if len(fname)>1:
        print('more than one file found') 
    
    return os.path.join(prefix, fname[0])

In [3]:
mask = xr.open_dataset('/home/jvalenti/MOAD/grid2/mesh_mask202108_TDV.nc')
coords=  xr.open_dataset('~/MOAD/grid/coordinates_seagrid_SalishSea201702.nc', decode_times=False)

In [4]:
runs = ['SHEM18','tuning/diat_pref','tuning/exc_hbac','tuning/exc_hbac_2','tuning/growth_flag','tuning/growth_flag_2','tuning/mort_hbac','tuning/pred_flag','tuning/remin','tuning/remin2','tuning/predmine','tuning/mort_hbac_2','tuning/remin2_l','tuning/predmine_z2']

In [5]:
jjii = xr.open_dataset('~/MOAD/grid/grid_from_lat_lon_mask999.nc')
def finder(lati,loni):
    j = [jjii.jj.sel(lats=lati, lons=loni, method='nearest').item()][0]
    i = [jjii.ii.sel(lats=lati, lons=loni, method='nearest').item()][0]
    return j,i

In [6]:
cnode = [49.04005171084712, -123.42548556523623]
jj, ii = finder(cnode[0], cnode[1])

In [7]:
start = dt.datetime(2018, 2, 28)
end = dt.datetime(2018, 7, 1)

folders = []
for i in range(0,(end - start).days + 1,5):
    folders.append((start + dt.timedelta(days=i)).strftime('%d%b%y').lower())
pathbase = '/home/jvalenti/scratch/run_SHEM/SSBase/'
for run in runs:
    path = f'/home/jvalenti/scratch/run_SHEM/{run}/'
    PPtot= []
    PPtot_diff = []
    for folder in folders:
        fn = make_filename(path,folder, var='prod_T', res='d')
        fbase = make_filename(pathbase,folder, var='prod_T', res='d')
        with xr.open_dataset(fn) as ds:
            PPdiat = ds['PPDIAT'].isel(time_counter=0, y=slice(jj-1,jj+2), x=slice(ii-1,ii+2))*mask['e3t_0'].isel(t=0, y=slice(jj-1,jj+2), x=slice(ii-1,ii+2))
            PPphy = ds['PPPHY'].isel(time_counter=0, y=slice(jj-1,jj+2), x=slice(ii-1,ii+2))*mask['e3t_0'].isel(t=0, y=slice(jj-1,jj+2), x=slice(ii-1,ii+2))
        with xr.open_dataset(fbase) as ds:
            PPdiat_b = ds['PPDIAT'].isel(time_counter=0, y=slice(jj-1,jj+2), x=slice(ii-1,ii+2))*mask['e3t_0'].isel(t=0, y=slice(jj-1,jj+2), x=slice(ii-1,ii+2))
            PPphy_b = ds['PPPHY'].isel(time_counter=0, y=slice(jj-1,jj+2), x=slice(ii-1,ii+2))*mask['e3t_0'].isel(t=0, y=slice(jj-1,jj+2), x=slice(ii-1,ii+2))
        PPtot_diff.append(PPdiat.sum()+PPphy.sum() - PPdiat_b.sum() - PPphy_b.sum())
        PPtot.append(PPdiat.sum()+PPphy.sum())
    print(run,np.mean(PPtot_diff))

SHEM18 -0.20070269865461499
tuning/diat_pref -0.18154445198399444
tuning/exc_hbac -0.2076492926819033
tuning/exc_hbac_2 -0.2085422483208362
tuning/growth_flag -0.26049749713287296
tuning/growth_flag_2 -0.1265252964419469
tuning/mort_hbac -0.2445340876014784
tuning/pred_flag -0.19112463768486865
tuning/remin -0.19014847971626106
tuning/remin2 -0.15657215557516585
tuning/predmine -0.08699289774384848
tuning/mort_hbac_2 -0.24793592229656988
tuning/remin2_l -0.15572554957552037
tuning/predmine_z2 -0.05843600742035072
